In [3]:
import pandas as pd
import json
from pathlib import Path
import os

# Set up paths
joke_books_dir = Path("/Users/mshen/Documents/AI_native/AI-native/FTeamProjectBackend/joke-dataset/joke_books")
print(f"Loading JSON files from: {joke_books_dir.absolute()}")


Loading JSON files from: /Users/mshen/Documents/AI_native/AI-native/FTeamProjectBackend/joke-dataset/joke_books


In [4]:
# Find all JSON files in the joke_books directory
json_files = sorted(joke_books_dir.glob("*.json"))
print(f"Found {len(json_files)} JSON files:")
for f in json_files:
    print(f"  - {f.name}")


Found 5 JSON files:
  - 100-KIDS-JOKES.json
  - Animal-Jokes-for-Kids-Printable.json
  - LOTS_OF_JOKES_FOR_KIDS.json
  - Math-Jokes-for-Kids.json
  - firecrawl_jokes.json


In [12]:
# Load all JSON files and create DataFrames
all_dataframes = []

for json_file in json_files:
    print(f"\nLoading: {json_file.name}")
    
    # Read JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        jokes = json.load(f)
    
    # Create DataFrame from the jokes
    df = pd.DataFrame(jokes)
    
    # Add Source column with the filename (without .json extension)
    df['Source'] = json_file.stem
    
    # Ensure we have the expected columns
    expected_columns = ['Question', 'Answer', 'Age Group', 'Scenario']
    for col in expected_columns:
        if col not in df.columns:
            print(f"  Warning: Missing column '{col}' in {json_file.name}")
    
    print(f"  Loaded {len(df)} jokes")
    
    # Remove duplicates within this source based on Question and Answer
    # Normalize text for comparison (lowercase, strip whitespace)
    df['Question_normalized'] = df['Question'].str.lower().str.strip()
    df['Answer_normalized'] = df['Answer'].str.lower().str.strip()
    
    # Remove duplicates, keeping the first occurrence
    initial_count = len(df)
    df = df.drop_duplicates(subset=['Question_normalized', 'Answer_normalized'], keep='first')
    
    # Remove the temporary normalization columns
    df = df.drop(columns=['Question_normalized', 'Answer_normalized'])
    
    duplicates_removed = initial_count - len(df)
    if duplicates_removed > 0:
        print(f"  Removed {duplicates_removed} duplicate(s), {len(df)} unique jokes remaining")
    else:
        print(f"  No duplicates found, {len(df)} jokes")
    
    all_dataframes.append(df)



Loading: 100-KIDS-JOKES.json
  Loaded 85 jokes
  No duplicates found, 85 jokes

Loading: Animal-Jokes-for-Kids-Printable.json
  Loaded 16 jokes
  No duplicates found, 16 jokes

Loading: LOTS_OF_JOKES_FOR_KIDS.json
  Loaded 517 jokes
  Removed 1 duplicate(s), 516 unique jokes remaining

Loading: Math-Jokes-for-Kids.json
  Loaded 39 jokes
  No duplicates found, 39 jokes

Loading: firecrawl_jokes.json
  Loaded 998 jokes
  Removed 105 duplicate(s), 893 unique jokes remaining


In [13]:
# Merge all DataFrames into one
merged_df = pd.concat(all_dataframes, ignore_index=True)

print(f"\nTotal jokes in merged DataFrame: {len(merged_df)}")
print(f"\nDataFrame shape: {merged_df.shape}")
print(f"\nColumns: {list(merged_df.columns)}")



Total jokes in merged DataFrame: 1549

DataFrame shape: (1549, 5)

Columns: ['Question', 'Answer', 'Age Group', 'Scenario', 'Source']


In [15]:
# Display first few rows
merged_df.head(10)


,Question,Answer,Age Group,Scenario,Source
0,Why did the computer go to the doctor?,It had a blue tooth.,5-8,"[home, school]",100-KIDS-JOKES
1,Why can’t your nose be 12 inches long?,Because then it would be a foot.,5-8,"[school, home]",100-KIDS-JOKES
2,How do you get a tissue to dance?,You put a little boogie in it.,5-8,"[school, home]",100-KIDS-JOKES
3,Why did they quit giving tests at the zoo?,Because it was full of cheetahs.,5-8,"[school, home]",100-KIDS-JOKES
4,Why is a bad joke like a pencil?,Because it has no point.,5-8,"[school, home]",100-KIDS-JOKES
5,Where do polar bears keep their money?,A snow bank.,5-8,"[home, vacation]",100-KIDS-JOKES
6,What room can no one enter?,A mushroom.,5-8,"[school, home]",100-KIDS-JOKES
7,What kind of key can never unlock a door?,A monkey.,5-8,"[school, home]",100-KIDS-JOKES
8,What has four wheels and flies?,A garbage truck.,5-8,"[home, school]",100-KIDS-JOKES
9,Why do graveyards have a fence around them?,Because people are dying to get in.,>12,"[home, party]",100-KIDS-JOKES


In [16]:
# Check data types and basic info
merged_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1549 entries, 0 to 1548
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Question   1549 non-null   object
 1   Answer     1549 non-null   object
 2   Age Group  1549 non-null   object
 3   Scenario   1549 non-null   object
 4   Source     1549 non-null   object
dtypes: object(5)
memory usage: 60.6+ KB


In [17]:
# Summary statistics by source
print("Jokes per source:")
print(merged_df['Source'].value_counts())
print(f"\nTotal: {len(merged_df)} jokes")


Jokes per source:
Source
firecrawl_jokes                    893
LOTS_OF_JOKES_FOR_KIDS             516
100-KIDS-JOKES                      85
Math-Jokes-for-Kids                 39
Animal-Jokes-for-Kids-Printable     16
Name: count, dtype: int64

Total: 1549 jokes


In [18]:
# Summary by Age Group
print("Distribution by Age Group:")
print(merged_df['Age Group'].value_counts())


Distribution by Age Group:
Age Group
5-8     705
8-12    535
>12     309
Name: count, dtype: int64


In [19]:
# Calculate total unique jokes across all sources
# Normalize Question and Answer for comparison
merged_df['Question_normalized'] = merged_df['Question'].str.lower().str.strip()
merged_df['Answer_normalized'] = merged_df['Answer'].str.lower().str.strip()

# Count unique jokes based on Question and Answer (across all sources)
unique_jokes_df = merged_df.drop_duplicates(subset=['Question_normalized', 'Answer_normalized'], keep='first')

# Remove temporary columns
unique_jokes_df = unique_jokes_df.drop(columns=['Question_normalized', 'Answer_normalized'])

total_jokes = len(merged_df)
unique_jokes_count = len(unique_jokes_df)
duplicates_across_sources = total_jokes - unique_jokes_count

print("="*60)
print("UNIQUE JOKES SUMMARY")
print("="*60)
print(f"Total jokes (with duplicates across sources): {total_jokes}")
print(f"Unique jokes (across all sources): {unique_jokes_count}")
print(f"Duplicates across sources: {duplicates_across_sources}")
print(f"\nPercentage unique: {(unique_jokes_count/total_jokes)*100:.1f}%")
print("="*60)


UNIQUE JOKES SUMMARY
Total jokes (with duplicates across sources): 1549
Unique jokes (across all sources): 1529
Duplicates across sources: 20

Percentage unique: 98.7%
